# Pipeline Dataset: Pengumpulan sampai StatistikNotebook ini membangun dataset verifikasi gambar-teks dari portal berita, mulai daripengambilan URL sampai tabel statistik. Satu berkas, dijalankan berurutan.**Tahap**1. Parameter dan utilitas2. Pengumpulan URL artikel3. Parsing halaman artikel4. Penyaringan dan deduplikasi5. Konstruksi tiga kelas (true, out-of-context, miscaptioned)6. Pembagian data (split per artikel)7. Statistik dataset**Keluaran** ada di `data/`:| Berkas | Isi ||---|---|| `articles.jsonl` | artikel yang lolos penyaringan || `pairs_v1.jsonl` | pasangan tiga kelas sebelum split || `pairs_locked.jsonl` | pasangan final dengan kolom `fold` |

## 0. ParameterUbah `LIMIT` untuk uji cepat. `LIMIT = 50` cukup untuk memastikan seluruh sel bekerja,`LIMIT = 3900` untuk menjalankan penuh.

In [ ]:
from pathlib import Pathimport calendar, datetime, html as ihtml, json, os, random, re, timeimport urllib.requestfrom collections import Counter, defaultdictfrom concurrent.futures import ThreadPoolExecutordef repo_root(start: Path | None = None) -> Path:    p = (start or Path.cwd()).resolve()    while p != p.parent:        if (p / ".git").exists():            return p        p = p.parent    return (start or Path.cwd()).resolve()ROOT = repo_root()OUT = ROOT / "data"OUT.mkdir(parents=True, exist_ok=True)LIMIT = 50          # 3900 untuk menjalankan penuhWORKERS = 8SEED = 42UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120 Safari/537.36"}rng = random.Random(SEED)print("root:", ROOT)print("keluaran:", OUT)print("limit:", LIMIT)

## 1. Utilitas HTTPSatu fungsi `get()` dengan percobaan ulang. Semua pengambilan halaman lewat sini,supaya waktu tunggu dan header konsisten.

In [ ]:
def get(url: str, retries: int = 2, timeout: int = 30) -> str:    for i in range(retries):        try:            req = urllib.request.Request(url, headers=UA)            with urllib.request.urlopen(req, timeout=timeout) as r:                return r.read().decode("utf8", "ignore")        except Exception:            if i == retries - 1:                raise            time.sleep(1)print(get("https://www.kompas.com/tag/banjir?page=1")[:80])

## 2. Pengumpulan URLHalaman tag dibaca, tautan artikel diambil dengan satu pola, lalu dideduplikasimemakai himpunan. Batas `LIMIT` dihentikan lebih awal supaya tidak mengambilseluruh arsip.

In [ ]:
ARTICLE = re.compile(r'href="(https://[a-z]+\.kompas\.com/read/\d{4}/\d{2}/\d{2}/[^"]+)"')def add_urls(urls: list[str], seen: set[str], url: str) -> int:    try:        page = get(url)    except Exception:        return 0    added = 0    for u in ARTICLE.findall(page):        if u not in seen:            seen.add(u)            urls.append(u)            added += 1    return added

In [ ]:
FRONTS = [    "https://regional.kompas.com/", "https://nasional.kompas.com/",    "https://megapolitan.kompas.com/", "https://tekno.kompas.com/",    "https://money.kompas.com/", "https://bola.kompas.com/",    "https://lifestyle.kompas.com/", "https://health.kompas.com/",    "https://edukasi.kompas.com/", "https://travel.kompas.com/",    "https://properti.kompas.com/", "https://otomotif.kompas.com/",    "https://sains.kompas.com/", "https://internasional.kompas.com/",]TAGS = [    "banjir", "kebakaran", "kecelakaan", "longsor", "gempa", "kriminal", "hukum",    "demo", "kesehatan", "pendidikan", "ekonomi", "politik", "pemilu", "olahraga",    "teknologi", "otomotif", "travel", "gaya-hidup", "properti", "internasional",    "jakarta", "surabaya", "bandung", "semarang", "medan", "makassar", "palembang",    "yogyakarta", "malang", "denpasar", "cuaca", "lingkungan", "energi", "pangan",    "transportasi", "infrastruktur", "kesehatan-masyarakat", "viral", "hoaks",    "bencana", "banjir-bandang", "tanah-longsor", "gempa-bumi", "kebakaran-hutan",    "kecelakaan-lalu-lintas", "kriminalitas", "korupsi", "pengadilan", "pemerintahan",    "kesejahteraan", "lingkungan-hidup", "pertanian", "perikanan", "perdagangan",]def collect(limit: int) -> list[str]:    urls, seen = [], set()    for f in FRONTS:        add_urls(urls, seen, f)        if len(urls) >= limit:            return urls[:limit]    for tag in TAGS:        for page in range(1, 21):            add_urls(urls, seen, f"https://www.kompas.com/tag/{tag}?page={page}")            if len(urls) >= limit:                return urls[:limit]    return urls[:limit]

In [ ]:
t0 = time.time()urls = collect(LIMIT)json.dump(urls, open(OUT / "urls.json", "w"), indent=0)print(f"URL terkumpul: {len(urls)} | unik: {len(set(urls))} | {time.time()-t0:.0f} detik")print(urls[0])

## 3. Parsing artikelEmpat field diambil: judul, gambar utama, caption, dan kanal. Caption dibersihkan daritag HTML dan entitas.Pemeriksaan mutu: `og:image` harus sama dengan gambar yang diberi caption di halaman.Foto utama dan foto ber-caption adalah elemen berbeda, dan hanya pasangan yang samayang layak dijadikan kelas `true`.

In [ ]:
def plain(s: str) -> str:    return re.sub(r"\s+", " ", ihtml.unescape(re.sub(r"<[^>]+>", " ", s))).strip()def parse(url: str) -> dict:    page = get(url)    title = re.search(r"<h1[^>]*>(.*?)</h1>", page, re.S)    og = re.search(r'property="og:image" content="([^"]+)"', page)    cap = re.search(r'<div class="photo__caption[^"]*"[^>]*>(.*?)</div>', page, re.S)    cat = re.search(r"//([a-z]+)\.kompas\.com/", url)    return {        "url": url,        "title": plain(title.group(1)) if title else "",        "image": og.group(1) if og else "",        "caption": plain(cap.group(1)) if cap else "",        "category": cat.group(1) if cat else "",    }

In [ ]:
def safe(u: str):    try:        return parse(u)    except Exception:        return NoneRAW = OUT / "articles_raw.jsonl"done = set()if RAW.exists():    done = {json.loads(l)["url"] for l in open(RAW)}todo = [u for u in urls if u not in done]print(f"sudah ada {len(done)} | sisa {len(todo)}")t0 = time.time()with open(RAW, "a") as f, ThreadPoolExecutor(max_workers=WORKERS) as ex:    for i, a in enumerate(ex.map(safe, todo), 1):        if a is not None:            f.write(json.dumps(a, ensure_ascii=False) + "\n")        if i % 100 == 0:            f.flush()            print(f"{i}/{len(todo)} | {time.time()-t0:.0f} detik")print("selesai parsing")

## 4. Penyaringan dan deduplikasiEnam kriteria, dijalankan berurutan. Duplikat gambar dideteksi setelah parameter`filters:` dibuang, karena banyak artikel memakai foto ilustrasi yang sama denganURL berbeda.

In [ ]:
SKIP_IMAGE = re.compile(r"(logo|default|placeholder|avatar|icon)", re.I)IMG_KEY = re.compile(r"/filters:[^/]+")arts, reject = [], Counter()for line in open(RAW):    a = json.loads(line)    if not a["image"]:        reject["tanpa gambar"] += 1    elif SKIP_IMAGE.search(a["image"]):        reject["gambar logo atau placeholder"] += 1    elif len(a["caption"].split()) < 6:        reject["caption terlalu pendek atau kosong"] += 1    elif len(a["title"].split()) < 4:        reject["judul terlalu pendek"] += 1    else:        arts.append(a)seen, uniq = set(), []for a in arts:    key = IMG_KEY.sub("", a["image"])    if key in seen:        reject["gambar duplikat"] += 1        continue    seen.add(key)    uniq.append(a)print(f"lolos filter: {len(uniq)}")for k, v in reject.most_common():    print(f"  {k}: {v}")with open(OUT / "articles.jsonl", "w") as f:    for a in uniq:        f.write(json.dumps(a, ensure_ascii=False) + "\n")

## 5. Konstruksi tiga kelas| Kelas | Aturan ||---|---|| true | gambar dan caption dari artikel yang sama || miscaptioned | caption yang sama, satu entitas terlihat diganti || out-of-context | caption artikel A dipasangkan dengan gambar artikel B dari kanal lain |Penggantian entitas hanya dilakukan pada kata yang bisa diverifikasi dari gambar, dandilewati jika kata itu diikuti nama diri atau membentuk frasa khusus.

In [ ]:
TERLIHAT = {    "bencana": ["banjir", "kebakaran", "tanah longsor", "longsor", "gempa bumi",                "gempa", "angin puting beliung", "kekeringan", "banjir bandang", "erupsi"],    "kendaraan": ["sepeda motor", "motor", "mobil", "truk", "bus", "kereta api",                  "kapal", "sepeda", "angkutan umum"],    "bangunan": ["rumah", "sekolah", "pasar", "kantor", "jembatan", "masjid", "gereja",                 "gudang", "toko", "terminal", "stasiun", "bandara", "pelabuhan",                 "posko", "tenda"],    "orang": ["polisi", "tentara", "petugas pemadam kebakaran", "petugas", "relawan",              "warga", "siswa", "guru", "pedagang", "nelayan", "petani", "sopir",              "pengemudi"],    "kegiatan": ["evakuasi", "penyaluran bantuan", "sosialisasi", "rapat", "unjuk rasa",                 "demonstrasi", "penertiban", "pembersihan"],}def swap_visible(caption: str) -> str | None:    for words in TERLIHAT.values():        for w in words:            for m in re.finditer(rf"\b{re.escape(w)}\b", caption, flags=re.I):                if re.match(r"\s+[A-Z]", caption[m.end():m.end() + 2]):                    continue                if w.lower() == "rumah" and caption[m.end():].lower().startswith(" sakit"):                    continue                others = [x for x in words if x != w                          and not re.search(rf"\b{re.escape(x)}\b", caption, re.I)]                if not others:                    continue                rep = others[0]                if m.group()[0].isupper():                    rep = rep[0].upper() + rep[1:]                return caption[:m.start()] + rep + caption[m.end():]    return None

In [ ]:
pairs = [{"label": "true", "image": a["image"], "caption": a["caption"],          "source": a["url"], "category": a["category"]} for a in uniq]misc = 0for a in uniq:    new = swap_visible(a["caption"])    if new:        misc += 1        pairs.append({"label": "miscaptioned", "image": a["image"], "caption": new,                      "source": a["url"], "category": a["category"]})by_cat = defaultdict(list)for a in uniq:    by_cat[a["category"]].append(a)ooc = 0for a in uniq:    pool = [b for b in by_cat[a["category"]]            if b["url"] != a["url"] and b["image"] != a["image"]]    if not pool:        pool = [b for b in uniq if b["url"] != a["url"] and b["image"] != a["image"]]    if pool:        b = rng.choice(pool)        pairs.append({"label": "out-of-context", "image": b["image"],                      "caption": a["caption"], "source": a["url"],                      "category": a["category"]})        ooc += 1print("artikel:", len(uniq), "| miscaptioned:", misc, "| out-of-context:", ooc)print("total pasangan:", len(pairs))with open(OUT / "pairs_v1.jsonl", "w") as f:    for p in pairs:        f.write(json.dumps(p, ensure_ascii=False) + "\n")

## 6. Pembagian dataSplit dilakukan **per artikel**, bukan per pasangan. Artikel dengan caption identikdisatukan lebih dulu, lalu dibagikan bergiliran ke lima fold.Pasangan out-of-context dibentuk ulang di dalam fold, sehingga tidak ada gambar ataucaption yang menyeberang antara latih dan uji.

In [ ]:
def make_folds(rows, k=5, seed=42):    arts, parent, cap_owner = {}, {}, {}    def find(x):        while parent[x] != x:            parent[x] = parent[parent[x]]            x = parent[x]        return x    def union(a, b):        ra, rb = find(a), find(b)        if ra != rb:            parent[rb] = ra    for r in rows:        a = arts.setdefault(r["source"], {"n": 0, "mc": 0})        a["n"] += 1        a["mc"] += int(r["label"] == "miscaptioned")        parent.setdefault(r["source"], r["source"])        if r["label"] == "true":            cap = r["caption"].strip().lower()            if cap in cap_owner:                union(cap_owner[cap], r["source"])            else:                cap_owner[cap] = r["source"]    comp = {}    for a in arts:        comp.setdefault(find(a), []).append(a)    items = list(comp.values())    for v in items:        rng.shuffle(v)    items.sort(key=lambda v: -sum(arts[a]["mc"] for a in v))    fold_of = {}    for i, v in enumerate(items):        for a in v:            fold_of[a] = i % k    return fold_of

In [ ]:
rows = [json.loads(l) for l in open(OUT / "pairs_v1.jsonl")]fold_of = make_folds(rows, k=5, seed=SEED)# bentuk ulang out-of-context di dalam foldby_fold = defaultdict(list)for r in rows:    if r["label"] == "true":        by_fold[fold_of[r["source"]]].append(r)locked = []for r in rows:    r = dict(r)    r["fold"] = fold_of[r["source"]]    if r["label"] == "out-of-context":        cand = by_fold[r["fold"]]        pool = [x for x in cand if x["image"] != r["image"]                and x.get("category") != r.get("category")]        if not pool:            pool = [x for x in cand if x["image"] != r["image"]]        pick = rng.choice(pool)        r["image"] = pick["image"]    locked.append(r)# pemeriksaan kebocoranimg_folds, leak_img = {}, 0for r in locked:    if r["image"] in img_folds and img_folds[r["image"]] != r["fold"]:        leak_img += 1    img_folds[r["image"]] = r["fold"]cap_folds, leak_cap = {}, 0for r in locked:    if r["caption"] in cap_folds and cap_folds[r["caption"]] != r["fold"]:        leak_cap += 1    cap_folds[r["caption"]] = r["fold"]print(f"pasangan: {len(locked)} | kebocoran gambar: {leak_img} | caption: {leak_cap}")with open(OUT / "pairs_locked.jsonl", "w") as f:    for r in locked:        f.write(json.dumps(r, ensure_ascii=False) + "\n")

## 7. Statistik datasetAngka di bawah ini yang dipakai di Bab III. Semua dihitung dari berkas final,bukan dari catatan terpisah.

In [ ]:
import statistics as stlabels = Counter(r["label"] for r in locked)folds = Counter(r["fold"] for r in locked)panjang = [len(r["caption"].split()) for r in locked if r["label"] == "true"]kanal = Counter(r["category"] for r in locked if r["label"] == "true")kurung = sum(1 for r in locked if r["label"] == "true" and "(" in r["caption"])print("Kelas")for k, v in labels.most_common():    print(f"  {k:16s} {v:5d}  {v/len(locked)*100:5.1f}%")print("\nFold")for k in sorted(folds):    print(f"  fold {k}: {folds[k]}")print("\nCaption")print(f"  rata-rata {st.mean(panjang):.1f} kata | median {st.median(panjang)} "      f"| min {min(panjang)} | maks {max(panjang)}")print(f"  memuat kredit foto: {kurung}/{len(panjang)}")print("\nKanal teratas")for k, v in kanal.most_common(6):    print(f"  {k:16s} {v}")

## Catatan reproduksi- `LIMIT` dan `SEED` menentukan hasil. Mengubah `LIMIT` mengubah komposisi pasangan.- Berkas antara (`articles_raw.jsonl`, `articles.jsonl`, `pairs_v1.jsonl`) disimpan,  sehingga tahap berikutnya bisa dijalankan ulang tanpa mengulang pengambilan.- Skrip lama yang menghasilkan dataset versi lanjutan disimpan di luar repo, karena  versi notebook ini hanya mencakup pembentukan berbasis aturan.